# Large Run Step 2: QC Inventory and QC Figures

Purpose: build or reuse full QC, create the observed/synthetic-overlap QC sidecar, and generate compact QC summaries without loading the full inventory into memory.

Outputs: `qc_trace_summary.csv`, full `qc_inventory.csv`, overlap `qc_inventory_overlap.parquet`, comparison-eligible records, QC summary tables, QC figures, and manual-review queue.


## Setup
Purpose: load config/output paths and shared driver settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")


## Helpers
Purpose: provide skip/status/Slurm helpers for this notebook.

Outputs: helper functions only.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve QC Outputs
Purpose: show whether full QC, overlap QC, and compact QC summaries already exist.

Outputs: status table only.


In [ ]:
from spatial_vtk.config.metrics import metrics_settings_from_config

metric_settings = metrics_settings_from_config()
event_station_path = resolve_output_path("event_station_records", kind="table", create_parent=True)
trace_qc_path = resolve_output_path("qc_trace_summary", kind="table", create_parent=True)
qc_inventory_path = resolve_output_path("qc_inventory", kind="table", create_parent=True)
qc_inventory_overlap_path = resolve_output_path("qc_inventory_overlap", kind="table", create_parent=True)
comparison_eligible_path = resolve_output_path("comparison_eligible_records", kind="table", create_parent=True)
manual_queue_path = resolve_output_path("manual_review_queue", kind="table", create_parent=True)
retention_path = resolve_output_path("qc_metric_pair_retention", kind="table", create_parent=True)
event_station_retention_path = resolve_output_path("qc_event_station_pair_retention", kind="table", create_parent=True)
post_qc_records_path = resolve_output_path("post_qc_records", kind="table", create_parent=True)
drop_causes_path = resolve_output_path("qc_drop_causes", kind="table", create_parent=True)
drop_causes_overlap_path = resolve_output_path("qc_drop_causes_overlap", kind="table", create_parent=True)

show_files([
    event_station_path,
    trace_qc_path,
    qc_inventory_path,
    qc_inventory_overlap_path,
    comparison_eligible_path,
    retention_path,
    event_station_retention_path,
    post_qc_records_path,
    drop_causes_path,
    drop_causes_overlap_path,
    manual_queue_path,
])


## Submit Full QC Build
Purpose: submit the checkpointed QC workflow when full QC or the overlap sidecar is missing. The job writes full QC first, then refreshes `qc_inventory_overlap.parquet`.

Outputs: Slurm script and job submission, or a manual `sbatch` command.


In [ ]:
if should_run(trace_qc_path, qc_inventory_path, qc_inventory_overlap_path):
    from spatial_vtk.qc.build.slurm import slurm_settings_from_config, submit_qc_slurm_job, write_qc_slurm_script
    settings = slurm_settings_from_config(cfg)
    script_path = slurm_dir / "step02_build_qc_inventory.slurm"
    write_qc_slurm_script(
        event_station_path,
        script_path,
        settings,
        config_path=config_path,
        trace_qc_output=trace_qc_path,
        qc_inventory_output=qc_inventory_path,
        qc_inventory_overlap_output=qc_inventory_overlap_path,
    )
    submit_script(script_path)
else:
    print("QC outputs already exist; skipping QC Slurm submission.")


## Build Missing Overlap Sidecar Only
Purpose: if full QC exists but the Parquet sidecar is missing or stale, create only `qc_inventory_overlap.parquet` in Slurm.

Outputs: refreshed overlap QC sidecar.


In [ ]:
if not qc_inventory_path.exists():
    print("Full QC inventory is not ready yet.")
elif should_run(qc_inventory_overlap_path):
    script = write_python_slurm_script(
        "step02_qc_overlap_sidecar.slurm",
        f"""
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.config.outputs import resolve_output_path
        from spatial_vtk.qc import write_qc_inventory_overlap_from_full

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        event_station_path = resolve_output_path("event_station_records", kind="table", cfg=cfg, create_parent=True)
        qc_inventory_path = resolve_output_path("qc_inventory", kind="table", cfg=cfg, create_parent=True)
        overlap_path = resolve_output_path("qc_inventory_overlap", kind="table", cfg=cfg, create_parent=True)
        write_qc_inventory_overlap_from_full(qc_inventory_path, event_station_path, overlap_path, scope="event", chunksize={QC_CHUNKSIZE}, overwrite=True, verbose=True)
        """,
        job_name="svtk-step02-overlap",
        walltime="12:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Overlap QC sidecar exists; skipping.")


## Submit Compact QC Summaries
Purpose: derive compact plotting and review tables from disk-backed QC inventories using chunked readers. This avoids loading full QC in the notebook.

Outputs: comparison-eligible records, retention summaries, drop-cause summaries, and manual-review queue.


In [ ]:
if not qc_inventory_overlap_path.exists():
    print(f"Overlap QC sidecar is not ready yet: {qc_inventory_overlap_path}")
elif should_run(comparison_eligible_path, retention_path, event_station_retention_path, post_qc_records_path, drop_causes_path, drop_causes_overlap_path, manual_queue_path):
    script = write_python_slurm_script(
        "step02_qc_summary_tables.slurm",
        f"""
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.config.outputs import resolve_output_path
        from spatial_vtk.io import load_output_table, write_output_table
        from spatial_vtk.qc import (
            build_event_station_pair_retention_table_from_qc_inventory,
            build_metric_pair_retention_table_from_qc_inventory,
            build_post_qc_record_table_from_qc_inventory,
            build_qc_drop_cause_table_from_qc_inventory,
            export_manual_review_queue_from_qc_inventory,
            filter_event_station_records_for_source_overlap,
            write_comparison_eligibility_from_qc_inventory,
        )

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        chunksize = {QC_CHUNKSIZE}
        event_stations = load_output_table("event_station_records")
        events = load_output_table("prepared_events")
        qc_inventory = resolve_output_path("qc_inventory", kind="table", cfg=cfg, create_parent=True)
        qc_overlap = resolve_output_path("qc_inventory_overlap", kind="table", cfg=cfg, create_parent=True)
        comparison = resolve_output_path("comparison_eligible_records", kind="table", cfg=cfg, create_parent=True)
        write_comparison_eligibility_from_qc_inventory(qc_overlap, comparison, chunksize=chunksize, overwrite={OVERWRITE!r}, verbose=True)
        retention = build_metric_pair_retention_table_from_qc_inventory(qc_overlap, chunksize=chunksize, verbose=True)
        write_output_table("qc_metric_pair_retention", retention)
        event_station_retention = build_event_station_pair_retention_table_from_qc_inventory(qc_overlap, chunksize=chunksize, verbose=True)
        write_output_table("qc_event_station_pair_retention", event_station_retention)
        overlap_records = filter_event_station_records_for_source_overlap(event_stations, scope="event")
        post_qc = build_post_qc_record_table_from_qc_inventory(overlap_records, events=events, qc_summary=qc_overlap, chunksize=chunksize, pair_retention=event_station_retention, verbose=True)
        write_output_table("post_qc_records", post_qc)
        write_output_table("qc_drop_causes", build_qc_drop_cause_table_from_qc_inventory(qc_inventory, chunksize=chunksize, verbose=True))
        write_output_table("qc_drop_causes_overlap", build_qc_drop_cause_table_from_qc_inventory(qc_overlap, chunksize=chunksize, verbose=True))
        export_manual_review_queue_from_qc_inventory(qc_inventory, chunksize=chunksize, overwrite={OVERWRITE!r}, verbose=True)
        """,
        job_name="svtk-step02-summaries",
        walltime="12:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Compact QC summary tables already exist; skipping.")


## Render QC Figures
Purpose: render QC figures from compact tables only. This cell does not read the full QC inventory.

Outputs: retention, retention heatmap, post-QC map, and drop-cause figures.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
required = [retention_path, event_station_retention_path, post_qc_records_path, drop_causes_path, drop_causes_overlap_path]
if not MAKE_FIGURES:
    print("Skipping figures. Set SVTK_MAKE_FIGURES=1 to render them.")
elif not all(path.exists() for path in required):
    print("Skipping figures until all compact QC tables exist.")
else:
    from spatial_vtk.visualize.qc import (
        plot_event_station_retention_heatmap,
        plot_post_qc_station_event_map,
        plot_qc_drop_cause_diagnostics,
        plot_retention_summary,
    )
    retention = load_output_table("qc_metric_pair_retention")
    event_station_retention = load_output_table("qc_event_station_pair_retention")
    post_qc_records = load_output_table("post_qc_records")
    drop_causes = load_output_table("qc_drop_causes")
    drop_causes_overlap = load_output_table("qc_drop_causes_overlap")
    plot_retention_summary(retention, title="QC Pair Retention Summary (Observed/Synthetic Event Overlap)", showfig=False, savefig=True)
    plot_event_station_retention_heatmap(event_station_retention, title="Post-QC Pair Retention by Event and Station", showfig=False, savefig=True)
    plot_post_qc_station_event_map(post_qc_records, add_basemap=True, showfig=False, savefig=True)
    plot_qc_drop_cause_diagnostics(drop_causes, title="QC Drop Causes (All Events)", showfig=False, savefig=True)
    plot_qc_drop_cause_diagnostics(drop_causes_overlap, resolve_output_path("qc_drop_cause_diagnostics_overlap", kind="figure", create_parent=True), title="QC Drop Causes (Observed/Synthetic Event Overlap)", showfig=False, savefig=True)
    print("Wrote QC figures.")
